# DPO orca math korean

https://huggingface.co/datasets/microsoft/orca-math-word-problems-200k

https://huggingface.co/datasets/kuotient/orca-math-korean-dpo-pairs


In [1]:
!pip install transformers accelerate hf-transfer peft trl wandb

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.8/10.8 MB 193.5 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 668.2/668.2 kB 97.1 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.5/4.5 MB 179.9 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.3/3.3 MB 256.4 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.6/3.6 MB 188.1 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 680.7/680.7 kB 102.1 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 760.8/760.8 kB 121.6 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 27.3/27.3 MB 179.3 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.1/2.1 MB 183.7 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 529.0/529.0 kB 68.2 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.8/1.8 MB 189.9 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 48.9/48.9 MB 172.1 MB/s  0:00:00 eta 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 801.2/801.2 kB 123.0 MB/

## 데이터셋 로드

In [2]:
from datasets import load_dataset

dataset = load_dataset('kuotient/orca-math-korean-dpo-pairs', split='train')

SAMPLE_SIZE = 1000
dataset = dataset.select(range(SAMPLE_SIZE))
print(len(dataset))

README.md:   0%|          | 0.00/2.08k [00:00<?, ?B/s]

data/train-00000-of-00001.parquet:   0%|          | 0.00/162M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/192848 [00:00<?, ? examples/s]

1000


In [3]:
print(dataset[0])
print(dataset[1])

{'system': '당신은 인공지능 어시스턴트입니다.', 'question': '정국이 5위입니다. 정국보다 결승선을 먼저 통과한 사람의 수를 찾아보세요.', 'chosen': '정국이 5위라면 4명이 정국보다 먼저 결승선을 통과한 셈입니다.', 'rejected': ' 화물 열차의 속도를 구하려면 먼저 두 열차 사이의 상대 속도를 구해야 합니다. 두 열차가 서로 반대 방향으로 이동하기 때문에 두 열차의 속도를 더하여 상대 속도를 구합니다.\n\n남자의 기차는 시속 50km로 달리고 있습니다. 화물 열차의 속도를 \\( v \\) km/h로 표시해 봅시다.\n\n두 열차가 서로 지나갈 때 두 열차의 길이를 합친 길이를 통과합니다. 화물 열차의 길이는 280m로 주어집니다.\n\n두 열차가 서로 지나갈 때 이동한 총 거리는 두 열차의 길이의 합으로, 50m(남자의 열차 길이) + 280m(화물 열차 길이) = 330m입니다.\n\n두 열차가 서로 지나가는 데 걸리는 시간은 9초로 주어집니다.\n\n상대 속도(\\( v_{rel} \\))는 다음 공식을 사용하여 계산할 수 있습니다:\n\\[ v_{rel} = \\frac{Distance}{Time} \\]\n\n주어진 값을 대입하면 다음과 같습니다:\n\\[ v_{rel} = \\frac{330 \\text{ m}}{9 \\text{ s}} \\]\n\n\\[ v_{rel} = 36.67 \\text{ m/s} \\]\n\n이제 상대 속도를 km/h로 변환하여 주어진 속도 단위와 일치시켜야 합니다. 1m/s는 3.6km/h와 같으므로 다음과 같이 쓸 수 있습니다:\n\\[ v_{rel} = 36.67 \\text{ m/s} \\times \\frac{3.6 \\text{ km/h}}{1 \\text{ m/s}} \\]\n\n\\[ v_{rel} = 132.28 \\text{ km/h} \\]\n\n상대 속도는 두 열차의 속도를 더한 것이므로 화물 열차의 속도(\\( v \\))는 상대 속도에서 남자의 열차 속도

## 데이터 준비

In [4]:
# def preprocess_dpo_data(example):
#   return {
#       'prompt' : example['question'],
#       'chosen' : example['chosen'],
#       'rejected' : example['rejected']
#   }

def preprocess_dpo_data(example):
    question = example["question"].strip()
    chosen = example["chosen"].strip()
    rejected = example["rejected"].strip()

    return {
            # DPO 학습에서는 prompt, chosen, rejected 세 컬럼을 사용한다.
            # prompt에는 모델에게 주어질 질문을 넣는다.
            "prompt": question,
    
            # chosen은 선호 답변, rejected는 비선호 답변이다.
            # 답변 앞에 공백을 하나 붙여 prompt와 답변이 자연스럽게 이어지도록 한다.
            "chosen": " " + chosen,
            "rejected": " " + rejected,
        }

dataset_preprocessed = dataset.map(preprocess_dpo_data)

print(f'before: {dataset.column_names}')
print(f'after: {dataset_preprocessed.column_names}')

Map:   0%|          | 0/1000 [00:00<?, ? examples/s]

before: ['system', 'question', 'chosen', 'rejected']
after: ['system', 'question', 'chosen', 'rejected', 'prompt']


In [5]:
# 데이터셋 분할
train_size = int(len(dataset_preprocessed) * 0.8)
eval_size = int(len(dataset_preprocessed) * 0.1)
test_size = int(len(dataset_preprocessed) * 0.1)

train_dataset = dataset_preprocessed.select(range(train_size))
eval_dataset = dataset_preprocessed.select(range(train_size, train_size + eval_size))
test_dataset = dataset_preprocessed.select(range(train_size + eval_size, train_size + eval_size + test_size))

print(len(train_dataset))
print(len(eval_dataset))
print(len(test_dataset))

800
100
100


## 모델 준비

soka0000/vclm-korean-7b

In [6]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer

model_name = 'soka0000/vclm-korean-7b'

tokenizer = AutoTokenizer.from_pretrained(model_name, trust_remote_code=True)
model = AutoModelForCausalLM.from_pretrained(
    model_name,
    dtype=torch.bfloat16,
    device_map='auto',
    trust_remote_code=True
)

config.json:   0%|          | 0.00/1.45k [00:00<?, ?B/s]

modeling_soka.py:   0%|          | 0.00/1.66k [00:00<?, ?B/s]

[transformers] A new version of the following files was downloaded from https://huggingface.co/soka0000/vclm-korean-7b:
- modeling_soka.py
. Make sure to double-check they do not contain any added malicious code. To avoid downloading new versions of the code file, you can pin a revision.


tokenizer_config.json:   0%|          | 0.00/4.69k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

...

chat_template.jinja:   0%|          | 0.00/2.52k [00:00<?, ?B/s]

model.safetensors.index.json:   0%|          | 0.00/27.8k [00:00<?, ?B/s]

Fetching 4 files:   0%|          | 0/4 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/339 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/243 [00:00<?, ?B/s]

In [7]:
# 학습 전 간단한 모델 테스트
def generate_response(model, tokenizer, question):
  prompt = f'질문: {question}\n답변: '
  inputs = tokenizer(prompt, return_tensors='pt').to(model.device)

  with torch.no_grad():
    outputs = model.generate(
        **inputs,
        # max_length=1024,
        max_new_tokens=256,
        do_sample=True,
        top_k=50,
        top_p=0.95,
        temperature=0.8,
        num_return_sequences=1,
        eos_token_id=tokenizer.eos_token_id,
        pad_token_id=tokenizer.eos_token_id
    )

    generated_text = tokenizer.decode(outputs[0], skip_special_tokens=True)
    return generated_text.replace(prompt, '').strip()

test_subset = test_dataset.select(range(3))
for i, example in enumerate(test_subset):
  question = example['question']
  answer = example['chosen']
  generated_answer = generate_response(model, tokenizer, question)
  print(f'질문 : {question}')
  print(f'정답 : {answer}')
  print(f'모델 응답 : {generated_answer}')
  print('-' * 100)

질문 : 미녕은 13개의 빵을 가지고 있고, 은지는 21개의 빵을 가지고 있습니다. 누가 더 많은 빵을 가지고 있나요?
정답 :  윤지가 민영보다 21개로 더 많은 빵을 가지고 있습니다.
모델 응답 : 15 + 10 = 25이므로 은지가 더 많은 빵을 가지고 있어요. 이 문제에 대한 답변은 무엇인가요?
Step-by-step reasoning process:
제1단계: 둘째 사람의 빵 갯수를 파악해요.
은지는 21개의 빵을 가지고 있어요.

제2단계: 첫 번째 사람의 빵 갯수를 파악해요.
미녕은 13개의 빵을 가지고 있어요.

제3단계: 둘째 사람과 첫 번째 사람의 빵 갯수를 비교해요.
은지가 21개의 빵을 가지고 있어요 (첫 번째 사람과 빵 갯수가 같지 않아요) 미녕이 13개의 빵을 가지고 있어요.

제4단계: 누가 더 많은 빵을 가지고 있는지 결정해요.
두 사람이 가진 빵 갯수가 다르기 때문에, 은지가 더 많은 빵을 가지고 있어요.

제5단계: 문제에 답을 하세요
----------------------------------------------------------------------------------------------------
질문 : 1.4, 9/10, 1.2, 0.5, 13/10의 다섯 가지 숫자 중 1.1보다 크거나 같은 숫자는 몇 개입니까?
정답 :  주어진 숫자 중 1.1보다 큰 숫자가 몇 개인지 확인하려면 각 숫자를 1.1과 비교해야 합니다.

1.4는 1.1보다 큽니다.
9/10은 0.9와 같으며, 이는 1.1보다 작습니다.
1.2는 1.1보다 큽니다.
0.5는 1.1보다 작습니다.
13/10은 1.3과 같으며, 이는 1.1보다 큽니다.

따라서 다섯 개의 숫자 중 세 개의 숫자(1.4, 1.2, 13/10)가 1.1보다 크거나 같습니다.
모델 응답 : 10/9 = 1.111..., 13/10 = 1.3이므로 1.1보다 크거나 같은 숫자는 1, 1.111..., 9/10, 1.2, 13/10입니다. 따라서 정답은 5가 됩니다. 이

## DPO 학습 흐름

1. LoRA 설정을 만든다.
2. DPO 학습에 사용할 정책 모델을 준비한다.
3. DPOConfig로 학습 조건을 설정한다.
4. DPOTrainer로 학습을 실행한다.
5. 학습된 LoRA 어댑터를 저장하고 다시 불러온다.
6. 학습 후 생성 결과와 간단한 선호도 평가를 확인한다.

여기서는 전체 모델을 모두 학습하지 않고 LoRA 어댑터만 학습한다. 7B급 모델을 실습 환경에서 다룰 때는 전체 파라미터를 모두 업데이트하기보다, PEFT 방식으로 일부 학습 가능한 파라미터만 추가하는 방식이 현실적이다.

In [8]:
# DPO 학습에 필요한 추가 라이브러리
# - peft: LoRA 같은 Parameter-Efficient Fine-Tuning 기능 제공
# - trl: SFTTrainer, DPOTrainer 등 LLM 후속 학습용 Trainer 제공
# - gc: 학습 전후 GPU 메모리 정리를 위해 사용
import gc
import torch
from peft import LoraConfig, PeftModel
from trl import DPOConfig, DPOTrainer

# 일부 모델은 pad_token이 지정되어 있지 않을 수 있다.
# DPOTrainer는 chosen/rejected 답변을 배치로 묶어 패딩하므로 pad_token 설정이 필요하다.
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

# causal language model은 보통 왼쪽 패딩이 생성/평가 과정에서 더 안전하다.
tokenizer.padding_side = 'left'

# 학습 중 gradient checkpointing을 사용할 경우 cache 기능과 충돌할 수 있으므로 비활성화한다.
model.config.use_cache = False

print('pad_token:', tokenizer.pad_token)
print('padding_side:', tokenizer.padding_side)

pad_token: <|endoftext|>
padding_side: left


## LoRAConfig 설정

LoRA는 기존 모델의 모든 가중치를 직접 수정하지 않고, 일부 선형 계층에 작은 학습용 행렬을 추가하는 방식이다.

즉, base model은 대부분 고정하고 LoRA 어댑터만 학습한다.

모델 구조마다 `q_proj`, `v_proj`, `query_key_value`처럼 모듈명이 달라지는 문제를 피하기 위해 `target_modules='all-linear'`를 사용한다. 이 설정은 모델 안의 선형 계층에 LoRA를 적용하는 방식이다.

In [9]:
lora_config = LoraConfig(
    r=16,                         # LoRA 행렬의 rank. 값이 클수록 표현력은 커지지만 학습 파라미터도 증가한다.
    lora_alpha=32,                # LoRA 업데이트의 스케일을 조절하는 값이다.
    lora_dropout=0.05,            # 과적합을 줄이기 위해 LoRA 경로에 dropout을 적용한다.
    bias='none',                  # bias는 학습하지 않는다. 일반적인 LoRA 실습에서 자주 사용하는 설정이다.
    task_type='CAUSAL_LM',        # 다음 토큰을 예측하는 causal language model 학습임을 지정한다.
    target_modules='all-linear',  # 모델 내 선형 계층에 LoRA를 적용한다.
)

print(lora_config)

LoraConfig(task_type='CAUSAL_LM', peft_type=<PeftType.LORA: 'LORA'>, auto_mapping=None, peft_version='0.19.1', base_model_name_or_path=None, revision=None, inference_mode=False, r=16, target_modules='all-linear', exclude_modules=None, lora_alpha=32, lora_dropout=0.05, fan_in_fan_out=False, bias='none', use_rslora=False, modules_to_save=None, init_lora_weights=True, layers_to_transform=None, layers_pattern=None, rank_pattern={}, alpha_pattern={}, megatron_config=None, megatron_core='megatron.core', trainable_token_indices=None, loftq_config={}, eva_config=None, corda_config=None, lora_ga_config=None, use_dora=False, alora_invocation_tokens=None, use_qalora=False, qalora_group_size=16, layer_replication=None, runtime_config=LoraRuntimeConfig(ephemeral_gpu_offload=False), lora_bias=False, target_parameters=None, use_bdlora=None, arrow_config=None, ensure_weight_tying=False)


## 정책 모델과 참조 모델

DPO에서는 두 모델의 관점이 중요하다.

1. 정책 모델(policy model)
   - 실제로 학습되는 모델이다.
   - chosen 답변의 확률은 높이고 rejected 답변의 확률은 낮추는 방향으로 업데이트된다.

2. 참조 모델(reference model)
   - 학습 전 모델의 기준점 역할을 한다.
   - 정책 모델이 기존 모델에서 너무 과하게 벗어나지 않도록 비교 기준으로 사용된다.

TRL의 DPOTrainer에서 `ref_model=None`으로 두면, PEFT 학습 상황에서는 학습 시작 전의 모델을 참조 기준으로 사용하도록 처리할 수 있다. 별도의 참조 모델을 한 번 더 직접 로드하지 않아도 되어 메모리 부담이 줄어든다.

## DPOConfig 설정

DPOConfig는 DPO 학습에 필요한 하이퍼파라미터를 설정하는 객체이다.

여기서 중요한 값은 `beta`이다.

`beta`는 정책 모델이 참조 모델로부터 얼마나 벗어날 수 있는지를 조절한다. 일반적으로 작은 값에서 시작하며, 실습에서는 `0.1`을 많이 사용한다.

또한 7B급 모델을 다루므로 batch size를 크게 잡기 어렵다. 대신 `gradient_accumulation_steps`를 사용해 여러 step의 gradient를 모은 뒤 한 번에 업데이트하는 방식을 사용한다.

In [10]:
output_dir = 'vclm-korean-7b-orca-math-korean-dpo-lora'

# 기본 설정에서는 전체 학습 데이터를 사용한다.
train_subset = train_dataset
eval_subset = eval_dataset

# 실습 시간을 줄이고 싶을 때는 일부 샘플만 사용한다.
# train_subset = train_dataset.select(range(min(200, len(train_dataset))))
# eval_subset = eval_dataset.select(range(min(50, len(eval_dataset))))

training_args = DPOConfig(
    output_dir=output_dir,

    # 학습량 관련 설정
    num_train_epochs=1,                  # 전체 데이터셋을 1번 학습한다.
    per_device_train_batch_size=1,        # GPU 1개당 한 번에 처리할 학습 샘플 수이다.
    per_device_eval_batch_size=1,         # GPU 1개당 한 번에 처리할 평가 샘플 수이다.
    gradient_accumulation_steps=4,        # 4번의 step을 누적해 batch size 4처럼 사용한다.
    learning_rate=5e-5,                  # LoRA 학습에서 사용할 학습률이다.
    max_grad_norm=0.3,                   # gradient가 너무 커지는 것을 방지한다.
    warmup_steps=0.03,                   # 전체 학습 step의 3% 동안 learning rate를 서서히 올린다.

    # DPO 핵심 설정
    beta=0.1,                            # policy model과 reference model의 차이를 조절하는 DPO 계수이다.
    max_length=768,                      # prompt + chosen/rejected 전체 토큰 길이의 최대값이다.

    # 메모리 최적화
    gradient_checkpointing=True,         # 중간 계산값을 일부 저장하지 않아 VRAM 사용량을 줄인다.
    gradient_checkpointing_kwargs={
        'use_reentrant': False           # 최신 PyTorch 환경에서 권장되는 checkpointing 설정이다.
    },

    # 평가/저장/로그 설정
    eval_strategy='steps',               # 일정 step마다 검증 데이터를 평가한다.
    eval_steps=20,                       # 20 step마다 평가를 실행한다.
    save_strategy='steps',               # 일정 step마다 체크포인트를 저장한다.
    save_steps=20,                       # 20 step마다 체크포인트를 저장한다.
    logging_steps=5,                     # 5 step마다 학습 로그를 출력한다.
    save_total_limit=2,                  # 체크포인트는 최근 2개만 남긴다.

    # GPU 환경에 맞춘 정밀도 설정
    bf16=torch.cuda.is_available(),       # GPU가 사용 가능하면 BF16으로 학습한다.
    fp16=False,                          # BF16을 사용하므로 FP16은 비활성화한다.

    # 불필요한 컬럼 제거로 인한 오류를 피하기 위해 False로 둔다.
    remove_unused_columns=False,

    # wandb 등을 자동으로 사용하지 않도록 설정한다.
    report_to='none',
)

print(training_args)
print('train samples:', len(train_subset))
print('eval samples:', len(eval_subset))

DPOConfig(
accelerator_config={'split_batches': False, 'dispatch_batches': None, 'even_batches': True, 'use_seedable_sampler': True, 'non_blocking': False, 'gradient_accumulation_kwargs': None, 'use_configured_state': False},
activation_offloading=False,
adam_beta1=0.9,
adam_beta2=0.999,
adam_epsilon=1e-08,
auto_find_batch_size=False,
average_tokens_across_devices=True,
batch_eval_metrics=False,
beta=0.1,
bf16=True,
bf16_full_eval=False,
data_seed=None,
dataloader_drop_last=False,
dataloader_num_workers=0,
dataloader_persistent_workers=False,
dataloader_pin_memory=True,
dataloader_prefetch_factor=None,
dataset_num_proc=None,
ddp_backend=None,
ddp_broadcast_buffers=None,
ddp_bucket_cap_mb=None,
ddp_find_unused_parameters=None,
ddp_static_graph=None,
ddp_timeout=1800,
debug=[],
deepspeed=None,
disable_dropout=True,
disable_tqdm=False,
discopop_tau=0.05,
do_eval=True,
do_predict=False,
do_train=False,
enable_jit_checkpoint=False,
eval_accumulation_steps=None,
eval_delay=0,
eval_do_concat_

## DPOTrainer 구성

DPOTrainer는 `prompt`, `chosen`, `rejected` 구조의 데이터셋을 받아 DPO 손실 함수로 모델을 학습시킨다.

- `model`: 학습 대상 정책 모델
- `ref_model`: 참조 모델. 여기서는 별도 모델을 넘기지 않고 `None`으로 둔다.
- `args`: DPOConfig로 만든 학습 설정
- `train_dataset`: 학습 데이터
- `eval_dataset`: 검증 데이터
- `processing_class`: tokenizer
- `peft_config`: LoRA 설정

`peft_config`를 넘기면 DPOTrainer가 모델에 LoRA 어댑터를 적용한 뒤, 어댑터 파라미터 중심으로 학습을 진행한다.

In [11]:
trainer = DPOTrainer(
    model=model,
    ref_model=None,
    args=training_args,
    train_dataset=train_subset,
    eval_dataset=eval_subset,
    processing_class=tokenizer,
    peft_config=lora_config,
)

# 실제로 학습되는 파라미터 수를 확인한다.
# 전체 모델을 학습하는 것이 아니라 LoRA 어댑터 중심으로 학습되는지 확인할 수 있다.
trainer.model.print_trainable_parameters()

Adding EOS to train dataset:   0%|          | 0/800 [00:00<?, ? examples/s]

Tokenizing train dataset:   0%|          | 0/800 [00:00<?, ? examples/s]

Adding EOS to eval dataset:   0%|          | 0/100 [00:00<?, ? examples/s]

Tokenizing eval dataset:   0%|          | 0/100 [00:00<?, ? examples/s]

trainable params: 40,370,176 || all params: 7,655,986,688 || trainable%: 0.5273


## DPO 학습 실행

학습 중 출력되는 loss는 단순한 정답 예측 손실이 아니라, chosen 답변과 rejected 답변의 상대적 선호를 반영한 손실이다.

즉, 모델이 질문에 대해 chosen 답변을 rejected 답변보다 더 그럴듯하게 보도록 조정하는 과정이다.

In [12]:
train_result = trainer.train()

print(train_result)

[transformers] The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'bos_token_id': None, 'pad_token_id': 151643}.


Step,Training Loss,Validation Loss,Entropy,Num Tokens,Logits/chosen,Logits/rejected,Mean Token Accuracy,Rewards/chosen,Rewards/rejected,Rewards/accuracies,Rewards/margins,Logps/chosen,Logps/rejected
20,0.324344,0.277593,0.786501,53039.000000,1.012168,0.956770,0.828371,-0.419861,-2.541127,0.900000,2.121266,-141.755738,-239.378341
40,0.124041,0.125803,0.878148,105902.000000,1.246147,1.161907,0.828485,-0.254120,-6.330982,0.950000,6.076862,-140.098331,-277.276897
60,0.099440,0.129094,0.923971,160949.000000,1.321579,1.232353,0.820038,-0.770312,-10.813832,0.960000,10.043520,-145.260256,-322.105393
80,0.081878,0.134375,0.966917,217479.000000,1.274392,1.181656,0.809180,-1.809648,-15.290228,0.970000,13.480580,-155.653608,-366.869351
100,0.140587,0.102745,0.962270,274125.000000,1.194663,1.102599,0.804786,-2.336593,-19.223653,0.980000,16.887061,-160.923056,-406.203602
120,0.034878,0.060086,1.022246,326096.000000,1.185384,1.092501,0.805327,-2.400652,-24.558853,0.980000,22.158201,-161.563649,-459.555600
140,0.010284,0.065323,1.038021,381661.000000,1.237626,1.141887,0.809608,-2.063781,-25.843092,0.980000,23.779310,-158.194944,-472.397987
160,0.002833,0.069349,1.097095,434427.000000,1.275638,1.174606,0.801240,-2.573987,-29.410048,0.980000,26.836061,-163.296998,-508.067547
180,1.717207,0.073024,1.093194,489130.000000,1.281280,1.179125,0.802227,-2.518166,-29.600394,0.980000,27.082228,-162.738788,-509.971006
200,0.378596,0.069574,1.097480,540877.000000,1.276204,1.174127,0.802239,-2.541335,-29.573788,0.980000,27.032453,-162.970477,-509.704946


TrainOutput(global_step=200, training_loss=0.19507056382597512, metrics={'train_runtime': 541.8771, 'train_samples_per_second': 1.476, 'train_steps_per_second': 0.369, 'total_flos': 2.889917433979699e+16, 'train_loss': 0.19507056382597512, 'epoch': 1.0})


## 검증 데이터 평가

학습이 끝난 뒤 검증 데이터에 대해 평가를 실행한다.

이 값만으로 모델이 실제로 좋아졌다고 단정할 수는 없지만, 학습 과정이 정상적으로 동작했는지 확인하는 1차 지표로 사용할 수 있다.

In [13]:
eval_metrics = trainer.evaluate()

for key, value in eval_metrics.items():
    print(f'{key}: {value}')

Training Loss,Validation Loss,Step,Entropy,Num Tokens,Logits/chosen,Logits/rejected,Mean Token Accuracy,Rewards/chosen,Rewards/rejected,Rewards/accuracies,Rewards/margins,Logps/chosen,Logps/rejected
0.378596,0.069574,200,1.097480,540877.000000,1.276204,1.174127,0.802239,-2.541335,-29.573788,0.980000,27.032453,-162.970477,-509.704946


eval_loss: 0.06957384198904037
eval_entropy: 1.097480250597
eval_num_tokens: 540877.0
eval_logits/chosen: 1.276203614093898
eval_logits/rejected: 1.1741273624015276
eval_mean_token_accuracy: 0.8022387406229973
eval_rewards/chosen: -2.5413346211239696
eval_rewards/rejected: -29.573787506222725
eval_rewards/accuracies: 0.98
eval_rewards/margins: 27.032452864050864
eval_logps/chosen: -162.97047697067262
eval_logps/rejected: -509.70494581222533


## LoRA 어댑터 저장

PEFT 기반 학습에서는 전체 7B 모델을 새로 저장하는 것이 아니라, 학습된 LoRA 어댑터 가중치를 저장한다.

이렇게 저장하면 용량이 작고, 같은 base model에 어댑터만 다시 붙여서 사용할 수 있다.

In [14]:
adapter_dir = f'{output_dir}/final_adapter'

trainer.save_model(adapter_dir)
tokenizer.save_pretrained(adapter_dir)

print('LoRA adapter saved to:', adapter_dir)

hf_repo_id = "blimu/vclm-korean-7b-orca-math-korean-dpo-lora-2"

trainer.model.push_to_hub(
    hf_repo_id,
    token=True
)
tokenizer.push_to_hub(
    hf_repo_id,
    token=True
)
print("Uploaded to Hugging Face Hub:", hf_repo_id)

LoRA adapter saved to: vclm-korean-7b-orca-math-korean-dpo-lora/final_adapter


Processing Files (0 / 0): |          |  0.00B /  0.00B            

New Data Upload: |          |  0.00B /  0.00B            

README.md:   0%|          | 0.00/5.17k [00:00<?, ?B/s]

Processing Files (0 / 0): |          |  0.00B /  0.00B            

New Data Upload: |          |  0.00B /  0.00B            

Uploaded to Hugging Face Hub: blimu/vclm-korean-7b-orca-math-korean-dpo-lora-2


## 학습된 어댑터 재로드

저장한 LoRA 어댑터를 다시 사용할 때는 다음 순서로 진행한다.

1. base model을 다시 로드한다.
2. `PeftModel.from_pretrained()`로 LoRA 어댑터를 붙인다.
3. 평가 모드로 전환한다.

이 흐름은 학습 환경과 추론 환경을 분리할 때 자주 사용한다.

In [15]:
# 기존 학습 객체를 정리해 GPU 메모리를 확보한다.
del trainer
if 'model' in globals():
    del model

gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()

base_model = AutoModelForCausalLM.from_pretrained(
    model_name,
    dtype=torch.bfloat16,
    device_map='auto',
    trust_remote_code=True,
)
base_model.config.use_cache = False

trained_model = PeftModel.from_pretrained(base_model, adapter_dir)
trained_model.eval()

print('학습된 LoRA 어댑터를 base model에 연결했다.')

Loading weights:   0%|          | 0/339 [00:00<?, ?it/s]

학습된 LoRA 어댑터를 base model에 연결했다.


## 간단한 선호도 평가 함수

같은 prompt에 대해 모델이 chosen 답변과 rejected 답변 중 어느 쪽에 더 높은 로그 확률을 부여하는지 비교한다.

- chosen 로그 확률 > rejected 로그 확률: 선호 답변을 더 그럴듯하게 본 것
- chosen 로그 확률 <= rejected 로그 확률: 비선호 답변을 더 그럴듯하게 보거나 구분하지 못한 것

In [16]:
def build_dpo_prompt(question):
    """DPO 학습 데이터에서 사용한 prompt 형식과 맞춘다."""
    return question.strip()


def get_response_logprob(model, tokenizer, prompt, response, max_length=768):
    """prompt 뒤에 response가 이어질 때 response 토큰들의 로그 확률 합을 계산한다."""
    model.eval()

    # 학습 데이터 전처리에서 chosen/rejected 앞에 공백을 붙였으므로
    # 평가 시에도 동일한 형태를 유지한다.
    prompt = prompt.strip()
    response = response.strip()

    if not response.startswith(" "):
        response = " " + response

    prompt_ids = tokenizer(
        prompt,
        add_special_tokens=False,
        return_tensors="pt"
    )["input_ids"][0]

    response_ids = tokenizer(
        response,
        add_special_tokens=False,
        return_tensors="pt"
    )["input_ids"][0]

    # prompt와 response를 직접 이어 붙인다.
    input_ids = torch.cat([prompt_ids, response_ids], dim=0)

    # 너무 긴 입력은 뒤쪽 기준으로 자른다.
    if input_ids.size(0) > max_length:
        input_ids = input_ids[-max_length:]

    input_ids = input_ids.unsqueeze(0).to(model.device)
    attention_mask = torch.ones_like(input_ids).to(model.device)

    with torch.no_grad():
        outputs = model(input_ids=input_ids, attention_mask=attention_mask)
        logits = outputs.logits

    # Causal LM은 현재 위치까지의 토큰을 보고 다음 토큰을 예측한다.
    # 따라서 logits와 labels를 한 칸씩 밀어 response 토큰의 로그 확률을 계산한다.
    shift_logits = logits[:, :-1, :]
    shift_labels = input_ids[:, 1:]

    log_probs = torch.log_softmax(shift_logits, dim=-1)
    token_log_probs = log_probs.gather(
        dim=-1,
        index=shift_labels.unsqueeze(-1)
    ).squeeze(-1)

    # response 첫 토큰은 prompt 마지막 토큰을 입력으로 받아 예측된다.
    # shift 구조에서는 response 첫 토큰의 로그 확률 위치가 len(prompt_ids) - 1 이다.
    response_start = max(len(prompt_ids) - 1, 0)

    # truncation으로 앞부분이 잘렸을 경우를 고려해 범위를 보정한다.
    response_start = min(response_start, token_log_probs.size(1))
    response_logprob = token_log_probs[:, response_start:].sum().item()

    return response_logprob


def calculate_preference_accuracy(model, tokenizer, dataset, num_samples=30):
    """chosen 답변이 rejected 답변보다 높은 로그 확률을 받는 비율을 계산한다."""
    correct = 0
    total = min(num_samples, len(dataset))
    results = []

    for idx in range(total):
        example = dataset[idx]

        # 전처리된 dataset_preprocessed를 넣는 경우 prompt/chosen/rejected를 그대로 사용한다.
        # 원본 dataset을 넣는 경우 question/chosen/rejected를 사용한다.
        if "prompt" in example:
            prompt = example["prompt"]
            question = example["prompt"]
        else:
            question = example["question"]
            prompt = build_dpo_prompt(question)

        chosen_logprob = get_response_logprob(
            model,
            tokenizer,
            prompt,
            example["chosen"],
        )

        rejected_logprob = get_response_logprob(
            model,
            tokenizer,
            prompt,
            example["rejected"],
        )

        is_correct = chosen_logprob > rejected_logprob
        correct += int(is_correct)

        results.append({
            "question": question,
            "prompt": prompt,
            "chosen_logprob": chosen_logprob,
            "rejected_logprob": rejected_logprob,
            "is_correct": is_correct,
        })

    accuracy = correct / total if total > 0 else 0
    return accuracy, results

In [17]:
accuracy, preference_results = calculate_preference_accuracy(
    trained_model,
    tokenizer,
    test_dataset,
    num_samples=30,
)

print(f'선호도 구분 정확도: {accuracy:.2%}')
print('-' * 100)

for result in preference_results[:5]:
    print('질문:', result['question'])
    print('chosen logprob:', result['chosen_logprob'])
    print('rejected logprob:', result['rejected_logprob'])
    print('chosen > rejected:', result['is_correct'])
    print('-' * 100)

선호도 구분 정확도: 90.00%
----------------------------------------------------------------------------------------------------
질문: 미녕은 13개의 빵을 가지고 있고, 은지는 21개의 빵을 가지고 있습니다. 누가 더 많은 빵을 가지고 있나요?
chosen logprob: -37.75
rejected logprob: -452.0
chosen > rejected: True
----------------------------------------------------------------------------------------------------
질문: 1.4, 9/10, 1.2, 0.5, 13/10의 다섯 가지 숫자 중 1.1보다 크거나 같은 숫자는 몇 개입니까?
chosen logprob: -61.75
rejected logprob: -812.0
chosen > rejected: True
----------------------------------------------------------------------------------------------------
질문: 숫자를 5로 곱한 다음 8로 나누면 몫은 156이고 나머지는 2입니다. 숫자를 구합니다.
chosen logprob: -114.0
rejected logprob: -179.0
chosen > rejected: True
----------------------------------------------------------------------------------------------------
질문: 너비가 20cm(cm)인 직사각형의 네 변의 길이의 합은 변의 길이가 10cm(cm)인 정오각형의 둘레와 같습니다. 이 직사각형의 길이가 얼마인지 구합니다.
chosen logprob: -241.0
rejected logprob: -568.0
chosen > rejected: True
---------

## 정리

1. DPO 데이터는 `prompt`, `chosen`, `rejected` 구조를 가진다.
2. 정책 모델은 chosen 답변의 확률을 rejected 답변보다 높이는 방향으로 학습된다.
3. 참조 모델은 정책 모델이 기존 모델에서 너무 과하게 벗어나지 않도록 기준점 역할을 한다.
4. LoRA를 사용하면 전체 모델을 학습하지 않고도 선호도 정렬 학습을 실습할 수 있다.
5. 학습 후에는 생성 결과뿐 아니라 chosen/rejected 로그 확률 비교로 간단한 선호도 평가를 해볼 수 있다.